# CRN: TD3 vs DDPG vs CAMO-TD3 — 3000 Episode Comparison

**Ramaiah Institute of Technology | Dr. Chitra M**  
Aditya Gangwani · Ryan Gomez · Shreya Revankar · Sneha Tapadar

### Features
- All three algorithms trained sequentially on Nakagami-*m* CRN environment
- **Auto-checkpoint every 300 episodes** → saved to Google Drive
- **Auto-resume** from latest checkpoint on restart — never lose progress
- Multi-page PDF comparison report generated at completion
- GPU-accelerated via PyTorch CUDA

## Step 1 — Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All outputs + checkpoints saved here — survives session restarts
DRIVE_DIR = '/content/drive/MyDrive/CRN_Training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_DIR}')


## Step 2 — Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'scipy', 'matplotlib'])
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Step 3 — Write Source Files

In [ ]:
%%writefile config.py
# =============================================================================
# config.py — All hyperparameters and constants in one place
# Cognitive Radio Network TD3 Simulation
# Ramaiah Institute of Technology, Bangalore
# =============================================================================

# ─── System Model ────────────────────────────────────────────────────────────
SIGMA2 = 1e-3        # AWGN noise power at each receiver (Watts)
P_P    = 1.0         # Primary Transmitter (PT) fixed transmit power (Watts)
P_MAX  = 3.0         # Maximum Secondary Transmitter (ST) power (Watts)

# ─── Reward Function Weights ─────────────────────────────────────────────────
ALPHA         = 1.0   # Weight for SU throughput (R_s)
BETA          =1.5  # Penalty weight for PU SINR constraint violation
GAMMA_REWARD  = 0.005  # Small penalty for energy usage (encourages efficiency)
SINR_THRESHOLD = 1.0  # Minimum acceptable SINR at PR (~0 dB)

# ─── RL / TD3 Hyperparameters ─────────────────────────────────────────────────
STATE_DIM           = 7        # [h_pp^2, h_sp^2, h_ss^2, h_ps^2, SINR_p, SINR_s, P_s_prev]
ACTION_DIM          = 1        # P_s (continuous, scalar)
HIDDEN_DIM          = 512      # Width of hidden layers in Actor and Critic

REPLAY_BUFFER_SIZE  = 200_000  # Maximum transitions stored
MIN_SAMPLES         = 1000     # Buffer must have this many before training starts

POLICY_NOISE        = 0.2      # Std of target policy smoothing noise (fraction of P_MAX)
NOISE_CLIP          = 0.5      # Clipping bound for target policy noise (fraction of P_MAX)
POLICY_DELAY        = 2        # Update actor only every N critic updates

EXPLORATION_NOISE_STD = 0.1    # Initial exploration noise std (fraction of P_MAX)
EXPLORATION_NOISE_END = 0.01   # Final exploration noise std (fraction of P_MAX)

BATCH_SIZE          = 256      # Mini-batch size for gradient updates
GRAD_UPDATES_PER_STEP = 2     # Gradient updates per env step (UTD ratio) — keeps GPU busy
LR_ACTOR            = 3e-4     # Adam learning rate for Actor
LR_CRITIC           = 3e-4     # Adam learning rate for Critics
GAMMA_DISCOUNT      = 0.99     # Discount factor for future rewards
TAU                 = 0.005    # Soft target network update rate (Polyak averaging)

TRAINING_EPISODES   = 7500     # Total training episodes
STEPS_PER_EPISODE   = 200      # Time steps per episode (channel coherence blocks)

# ─── Nakagami-m Fading Channel ───────────────────────────────────────────────
NAKAGAMI_M     = 3.0   # Fading severity (m=1 → Rayleigh; m=3 → moderate Nakagami fading)
NAKAGAMI_OMEGA = 1.0   # Mean power per link (Ω)

# ─── CAMO-TD3 (Constrained Adaptive Multi-Objective TD3) ─────────────────
GRU_HIDDEN_SIZE    = 64       # GRU hidden state width
GRU_NUM_LAYERS     = 2        # Number of GRU layers
BELIEF_DIM         = 16       # Compressed belief vector dimension
SEQ_LEN            = 8        # Observation history length for GRU encoder

LAMBDA1_INIT       = 3.0      # Initial Lagrangian weight for throughput objective
LAMBDA2_INIT       = 1.0      # Initial Lagrangian weight for interference constraint
LAMBDA3_INIT       = 0.01     # Initial Lagrangian weight for energy objective
LR_LAMBDA          = 0.0005   # Learning rate for Lagrangian multiplier updates
LAMBDA_MIN         = 0.1      # Floor: prevents any objective from being ignored
LAMBDA_MAX         = 20.0     # Ceiling: prevents runaway constraint weighting

MU_BIAS_INIT       = -0.05    # Directional noise bias (negative = lower power to protect PU)
NOISE_DECAY_STEPS  = 200_000  # Steps over which directional noise decays
VIOLATION_WINDOW   = 100      # Rolling window for constraint violation tracking

CAMO_HIDDEN_DIM    = 512      # Width of hidden layers in CAMO Actor and Critics

# ─── WebSocket / Server ───────────────────────────────────────────────────────
WS_HOST            = "0.0.0.0"
WS_PORT            = 8000
BROADCAST_INTERVAL = 50    # Broadcast every N environment steps
SCATTER_WINDOW     = 200   # Number of (SINR_dB, BER) pairs kept for scatter chart
OUTAGE_WINDOW      = 500   # Rolling window size for outage probability estimate


In [ ]:
%%writefile environment.py
# =============================================================================
# environment.py — CRN Environment (Gym-like, no gymnasium dependency)
#
# System model:
#   - 4 nodes: Primary Transmitter (PT), Primary Receiver (PR),
#              Secondary Transmitter (ST), Secondary Receiver (SR)
#   - Nakagami-m fading: |h|^2 ~ Gamma(m, Omega/m), drawn fresh every time step
#     (m=1 recovers Rayleigh/Exponential exactly)
#   - PT transmits at fixed power P_p; ST power P_s is the TD3 action
#   - SINR_p = (P_p * h_pp^2) / (P_s * h_sp^2 + sigma^2)
#   - SINR_s = (P_s * h_ss^2) / (P_p * h_ps^2 + sigma^2)
#   - Reward  = alpha*R_s - beta*max(0, thr - SINR_p) - gamma*(P_s/P_max)
# =============================================================================

from __future__ import annotations
import numpy as np
from dataclasses import dataclass, field
from config import (
    SIGMA2, P_P, P_MAX, SINR_THRESHOLD,
    ALPHA, BETA, GAMMA_REWARD,
    STATE_DIM, STEPS_PER_EPISODE,
    NAKAGAMI_M, NAKAGAMI_OMEGA,
)


@dataclass
class StepResult:
    state:  np.ndarray   # shape (7,) — next observation
    reward: float
    done:   bool
    info:   dict         # sinr_p, sinr_s, r_s, p_s, h_pp, h_sp, h_ss, h_ps


class CRNEnvironment:
    """
    Cognitive Radio Network environment.

    State vector (7-dimensional):
        [h_pp^2, h_sp^2, h_ss^2, h_ps^2, SINR_p, SINR_s, P_s_prev]

    Action:
        P_s — scalar float in [0, P_max], chosen by the TD3 agent.

    Reward:
        r = alpha * R_s
            - beta  * max(0, SINR_threshold - SINR_p)   # constraint violation penalty
            - gamma * (P_s / P_max)                      # energy efficiency penalty
    """

    def __init__(
        self,
        p_max:             float = P_MAX,
        p_p:               float = P_P,
        sigma2:            float = SIGMA2,
        sinr_threshold:    float = SINR_THRESHOLD,
        steps_per_episode: int   = STEPS_PER_EPISODE,
        alpha:             float = ALPHA,
        beta:              float = BETA,
        gamma_r:           float = GAMMA_REWARD,
        nakagami_m:        float = NAKAGAMI_M,
        nakagami_omega:    float = NAKAGAMI_OMEGA,
        seed:              int | None = None,
    ):
        self.p_max             = p_max
        self.p_p               = p_p
        self.sigma2            = sigma2
        self.sinr_threshold    = sinr_threshold
        self.steps_per_episode = steps_per_episode
        self.alpha             = alpha
        self.beta              = beta
        self.gamma_r           = gamma_r
        self.nakagami_m        = nakagami_m
        self.nakagami_omega    = nakagami_omega

        # Reproducible RNG (independent of global numpy state)
        self.rng = np.random.default_rng(seed)

        # Episode tracking
        self._step_count: int   = 0
        self._p_s_prev:   float = 0.0

        # Last channel gains (stored so GUI can read them)
        self._h_pp_sq: float = 0.0
        self._h_sp_sq: float = 0.0
        self._h_ss_sq: float = 0.0
        self._h_ps_sq: float = 0.0

    # ──────────────────────────────────────────────────────────────────────────
    # Public API
    # ──────────────────────────────────────────────────────────────────────────

    def reset(self) -> np.ndarray:
        """
        Start a new episode.
        Draws fresh channel gains and returns the initial 7D state.
        """
        self._step_count = 0
        self._p_s_prev   = 0.0

        h_pp, h_sp, h_ss, h_ps = self._draw_channels()
        sinr_p, sinr_s = self._compute_sinr(h_pp, h_sp, h_ss, h_ps, self._p_s_prev)

        return self._build_state(h_pp, h_sp, h_ss, h_ps, sinr_p, sinr_s, self._p_s_prev)

    def step(self, action: float) -> StepResult:
        """
        Execute one time step.

        Args:
            action: P_s value — the agent's chosen transmit power.
                    Should already be in [0, P_max]; we clip defensively.

        Returns:
            StepResult with next state, reward, done flag, and info dict.
        """
        # Clip action to valid range
        p_s = float(np.clip(action, 0.0, self.p_max))

        # Draw fresh Rayleigh fading channel gains (block-fading model)
        h_pp, h_sp, h_ss, h_ps = self._draw_channels()

        # Compute SINRs and throughput
        sinr_p, sinr_s = self._compute_sinr(h_pp, h_sp, h_ss, h_ps, p_s)
        r_s = float(np.log2(1.0 + sinr_s))          # SU throughput (bits/s/Hz)

        # Compute reward
        reward = self._compute_reward(sinr_p, sinr_s, p_s)

        # Advance counters
        self._step_count += 1
        done = (self._step_count >= self.steps_per_episode)

        # Build next state (stores current p_s as p_s_prev for next step)
        next_state = self._build_state(h_pp, h_sp, h_ss, h_ps, sinr_p, sinr_s, p_s)
        self._p_s_prev = p_s

        info = {
            "sinr_p": sinr_p,
            "sinr_s": sinr_s,
            "r_s":    r_s,
            "p_s":    p_s,
            "h_pp":   h_pp,
            "h_sp":   h_sp,
            "h_ss":   h_ss,
            "h_ps":   h_ps,
        }

        return StepResult(state=next_state, reward=reward, done=done, info=info)

    # ──────────────────────────────────────────────────────────────────────────
    # Private helpers
    # ──────────────────────────────────────────────────────────────────────────

    def _draw_channels(self) -> tuple[float, float, float, float]:
        """
        Draw Nakagami-m fading channel power gains.
        |h|^2 ~ Gamma(shape=m, scale=Omega/m) for each link independently.
        m=1 exactly recovers Rayleigh (Exponential(Omega)).

        Returns: (h_pp_sq, h_sp_sq, h_ss_sq, h_ps_sq)
        """
        scale = self.nakagami_omega / self.nakagami_m
        h_pp_sq = float(self.rng.gamma(self.nakagami_m, scale))   # PT → PR
        h_sp_sq = float(self.rng.gamma(self.nakagami_m, scale))   # ST → PR
        h_ss_sq = float(self.rng.gamma(self.nakagami_m, scale))   # ST → SR
        h_ps_sq = float(self.rng.gamma(self.nakagami_m, scale))   # PT → SR

        self._h_pp_sq = h_pp_sq
        self._h_sp_sq = h_sp_sq
        self._h_ss_sq = h_ss_sq
        self._h_ps_sq = h_ps_sq

        return h_pp_sq, h_sp_sq, h_ss_sq, h_ps_sq

    def _compute_sinr(
        self,
        h_pp_sq: float, h_sp_sq: float,
        h_ss_sq: float, h_ps_sq: float,
        p_s:     float,
    ) -> tuple[float, float]:
        """
        Compute SINR at Primary Receiver and Secondary Receiver.

        SINR_p = (P_p * h_pp^2) / (P_s * h_sp^2 + sigma^2)
        SINR_s = (P_s * h_ss^2) / (P_p * h_ps^2 + sigma^2)

        Denominator is always > 0 because sigma^2 > 0.
        """
        sinr_p = (self.p_p * h_pp_sq) / (p_s * h_sp_sq + self.sigma2)
        sinr_s = (p_s * h_ss_sq) / (self.p_p * h_ps_sq + self.sigma2)
        return float(sinr_p), float(sinr_s)

    def _compute_reward(self, sinr_p: float, sinr_s: float, p_s: float) -> float:
        """
        Reward = alpha * R_s
                 - beta  * max(0, SINR_threshold - SINR_p)
                 - gamma * (P_s / P_max)

        Positive component: SU throughput (log2(1 + SINR_s))
        Negative component 1: heavy penalty when PU SINR drops below threshold
        Negative component 2: small energy-use penalty
        """
        r_s     = float(np.log2(1.0 + sinr_s))
        penalty = max(0.0, self.sinr_threshold - sinr_p)
        energy  = p_s / self.p_max

        return self.alpha * r_s - self.beta * penalty - self.gamma_r * energy

    def _build_state(
        self,
        h_pp_sq: float, h_sp_sq: float,
        h_ss_sq: float, h_ps_sq: float,
        sinr_p:  float, sinr_s: float,
        p_s_prev: float,
    ) -> np.ndarray:
        """
        Assemble the 7-dimensional state vector as float32.

        State: [h_pp^2, h_sp^2, h_ss^2, h_ps^2, SINR_p, SINR_s, P_s_prev]
        """
        state = np.array(
            [h_pp_sq, h_sp_sq, h_ss_sq, h_ps_sq, sinr_p, sinr_s, p_s_prev],
            dtype=np.float32,
        )
        return state

    # ──────────────────────────────────────────────────────────────────────────
    # Properties
    # ──────────────────────────────────────────────────────────────────────────

    @property
    def step_count(self) -> int:
        return self._step_count

    @property
    def observation_space_dim(self) -> int:
        return STATE_DIM

    @property
    def action_space_dim(self) -> int:
        return 1


In [ ]:
%%writefile td3.py
# =============================================================================
# td3.py — Twin Delayed Deep Deterministic Policy Gradient (TD3)
#
# Implements the TD3 algorithm from Fujimoto et al., 2018:
#   "Addressing Function Approximation Error in Actor-Critic Methods"
#
# Key TD3 features implemented:
#   1. Twin critics  — two independent Q-networks; use min for target
#   2. Delayed policy updates — update actor every POLICY_DELAY critic steps
#   3. Target policy smoothing — add clipped Gaussian noise to target actions
#   4. Soft target updates  — Polyak averaging with tau
# =============================================================================

from __future__ import annotations
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import (
    STATE_DIM, ACTION_DIM, HIDDEN_DIM, P_MAX,
    REPLAY_BUFFER_SIZE, MIN_SAMPLES,
    LR_ACTOR, LR_CRITIC,
    GAMMA_DISCOUNT, TAU,
    POLICY_NOISE, NOISE_CLIP, POLICY_DELAY,
    BATCH_SIZE,
)


# =============================================================================
# Replay Buffer
# =============================================================================

class ReplayBuffer:
    """
    Circular replay buffer storing (s, a, r, s', done) transitions.
    Pre-allocates numpy arrays for efficiency — no list.append overhead.
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        action_dim: int = ACTION_DIM,
        max_size:   int = REPLAY_BUFFER_SIZE,
        device:     str = "cpu",
    ):
        self._max_size  = max_size
        self._device    = torch.device(device)
        self._ptr       = 0    # write pointer (circular)
        self._size      = 0    # number of valid entries

        # Pre-allocate directly on target device — eliminates batch CPU->GPU
        # transfer at sample time (only per-step single-element writes cross the bus)
        self._states      = torch.zeros((max_size, state_dim),  dtype=torch.float32, device=self._device)
        self._actions     = torch.zeros((max_size, action_dim), dtype=torch.float32, device=self._device)
        self._rewards     = torch.zeros((max_size, 1),          dtype=torch.float32, device=self._device)
        self._next_states = torch.zeros((max_size, state_dim),  dtype=torch.float32, device=self._device)
        self._dones       = torch.zeros((max_size, 1),          dtype=torch.float32, device=self._device)

    def add(
        self,
        state:      np.ndarray,
        action:     np.ndarray,
        reward:     float,
        next_state: np.ndarray,
        done:       bool,
    ) -> None:
        self._states[self._ptr]      = torch.as_tensor(state,               dtype=torch.float32)
        self._actions[self._ptr]     = torch.as_tensor(action,              dtype=torch.float32)
        self._rewards[self._ptr, 0]  = float(reward)
        self._next_states[self._ptr] = torch.as_tensor(next_state,          dtype=torch.float32)
        self._dones[self._ptr, 0]    = float(done)

        self._ptr  = (self._ptr + 1) % self._max_size
        self._size = min(self._size + 1, self._max_size)

    def sample(self, batch_size: int = BATCH_SIZE) -> tuple[torch.Tensor, ...]:
        """
        Sample a random mini-batch.
        Tensors are already on device — no transfer overhead.
        """
        idx = torch.randint(0, self._size, (batch_size,), device=self._device)
        return (
            self._states[idx],
            self._actions[idx],
            self._rewards[idx],
            self._next_states[idx],
            self._dones[idx],
        )

    def __len__(self) -> int:
        return self._size

    @property
    def is_ready(self) -> bool:
        return self._size >= MIN_SAMPLES


# =============================================================================
# Neural Network Modules
# =============================================================================

class Actor(nn.Module):
    """
    Policy network π(s) → P_s ∈ [0, P_max].

    Architecture:
        Linear(state_dim → hidden_dim) → ReLU
        Linear(hidden_dim → hidden_dim) → ReLU
        Linear(hidden_dim → action_dim) → Sigmoid × P_max

    Sigmoid ensures output is always in (0, P_max) without explicit clipping.
    """

    def __init__(
        self,
        state_dim:  int   = STATE_DIM,
        hidden_dim: int   = HIDDEN_DIM,
        action_dim: int   = ACTION_DIM,
        p_max:      float = P_MAX,
    ):
        super().__init__()
        self.p_max = p_max
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Sigmoid(),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.net(state) * self.p_max


class Critic(nn.Module):
    """
    Q-value network Q(s, a) → ℝ.

    Input: concatenation of [state, action]  (shape: batch × (state_dim + action_dim))

    Architecture:
        Linear(state_dim + action_dim → hidden_dim) → ReLU
        Linear(hidden_dim → hidden_dim) → ReLU
        Linear(hidden_dim → 1)
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        action_dim: int = ACTION_DIM,
        hidden_dim: int = HIDDEN_DIM,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        x = torch.cat([state, action], dim=-1)
        return self.net(x)


# =============================================================================
# TD3 Agent
# =============================================================================

class TD3Agent:
    """
    Twin Delayed Deep Deterministic Policy Gradient agent.

    Maintains:
        actor, actor_target
        critic1, critic1_target
        critic2, critic2_target
        ... and their Adam optimizers.
    """

    def __init__(
        self,
        state_dim:    int   = STATE_DIM,
        action_dim:   int   = ACTION_DIM,
        p_max:        float = P_MAX,
        lr_actor:     float = LR_ACTOR,
        lr_critic:    float = LR_CRITIC,
        gamma:        float = GAMMA_DISCOUNT,
        tau:          float = TAU,
        policy_noise: float = POLICY_NOISE,
        noise_clip:   float = NOISE_CLIP,
        policy_delay: int   = POLICY_DELAY,
        device:       str   = "auto",
    ):
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.p_max  = p_max

        # ── Actor + target ──────────────────────────────────────────────────
        self.actor        = Actor(state_dim, HIDDEN_DIM, action_dim, p_max).to(device)
        self.actor_target = Actor(state_dim, HIDDEN_DIM, action_dim, p_max).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)

        # ── Critic 1 + target ───────────────────────────────────────────────
        self.critic1        = Critic(state_dim, action_dim, HIDDEN_DIM).to(device)
        self.critic1_target = Critic(state_dim, action_dim, HIDDEN_DIM).to(device)
        self.critic1_target.load_state_dict(self.critic1.state_dict())
        self.critic1_optimizer = torch.optim.Adam(self.critic1.parameters(), lr=lr_critic)

        # ── Critic 2 + target ───────────────────────────────────────────────
        self.critic2        = Critic(state_dim, action_dim, HIDDEN_DIM).to(device)
        self.critic2_target = Critic(state_dim, action_dim, HIDDEN_DIM).to(device)
        self.critic2_target.load_state_dict(self.critic2.state_dict())
        self.critic2_optimizer = torch.optim.Adam(self.critic2.parameters(), lr=lr_critic)

        # ── TD3 hyperparameters ─────────────────────────────────────────────
        self.gamma        = gamma
        self.tau          = tau
        self.policy_noise = policy_noise * p_max   # scale noise to action range
        self.noise_clip   = noise_clip   * p_max   # scale clip  to action range
        self.policy_delay = policy_delay

        # Internal step counter (used for delayed policy updates)
        self._train_iterations = 0

        # Latest losses for GUI display
        self.last_critic1_loss: float = 0.0
        self.last_critic2_loss: float = 0.0
        self.last_actor_loss:   float | None = None

    # ──────────────────────────────────────────────────────────────────────────
    # Action selection
    # ──────────────────────────────────────────────────────────────────────────

    def select_action(
        self,
        state:             np.ndarray,
        exploration_noise: float = 0.0,
    ) -> float:
        """
        Deterministic action from actor + optional Gaussian exploration noise.

        Args:
            state: (STATE_DIM,) float32 numpy array
            exploration_noise: std of Gaussian noise added for exploration

        Returns:
            Scalar float in [0, P_max].
        """
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            action = self.actor(s).cpu().numpy().flatten()[0]

        if exploration_noise > 0.0:
            action += np.random.normal(0.0, exploration_noise)

        return float(np.clip(action, 0.0, self.p_max))

    # ──────────────────────────────────────────────────────────────────────────
    # Training step
    # ──────────────────────────────────────────────────────────────────────────

    def train_step(
        self,
        replay_buffer: ReplayBuffer,
        batch_size:    int = BATCH_SIZE,
    ) -> dict | None:
        """
        Perform one TD3 update.

        Returns a dict with loss values, or None if buffer isn't ready yet.

        TD3 update algorithm:
            1. Sample mini-batch from replay buffer
            2. Compute target actions with clipped noise (target policy smoothing)
            3. Compute target Q = r + γ(1-done) * min(Q1_tgt, Q2_tgt)
            4. Update Critic 1 and Critic 2 via MSE loss
            5. Every POLICY_DELAY steps:
               a. Update actor: minimize -Q1(s, π(s))
               b. Soft-update all four target networks
        """
        if not replay_buffer.is_ready:
            return None

        self._train_iterations += 1
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        # ── Step 1: Compute target actions with target policy smoothing ──────
        with torch.no_grad():
            # Add clipped Gaussian noise to smooth the target policy
            noise = torch.FloatTensor(actions.shape).normal_(0, self.policy_noise).to(self.device)
            noise = noise.clamp(-self.noise_clip, self.noise_clip)   # clip noise first

            target_actions = (self.actor_target(next_states) + noise).clamp(0.0, self.p_max)

            # ── Step 2: Compute target Q-values ─────────────────────────────
            q1_target = self.critic1_target(next_states, target_actions)
            q2_target = self.critic2_target(next_states, target_actions)
            q_target  = rewards + self.gamma * (1.0 - dones) * torch.min(q1_target, q2_target)

        # ── Step 3: Update Critic 1 ──────────────────────────────────────────
        q1_current = self.critic1(states, actions)
        loss_c1    = F.mse_loss(q1_current, q_target)
        self.critic1_optimizer.zero_grad()
        loss_c1.backward()
        self.critic1_optimizer.step()

        # ── Step 4: Update Critic 2 ──────────────────────────────────────────
        q2_current = self.critic2(states, actions)
        loss_c2    = F.mse_loss(q2_current, q_target)
        self.critic2_optimizer.zero_grad()
        loss_c2.backward()
        self.critic2_optimizer.step()

        self.last_critic1_loss = loss_c1.item()
        self.last_critic2_loss = loss_c2.item()
        actor_loss_val         = None

        # ── Step 5: Delayed policy update ────────────────────────────────────
        if self._train_iterations % self.policy_delay == 0:
            # Actor loss: maximize Q1(s, π(s))  →  minimize -Q1(s, π(s))
            actor_actions = self.actor(states)
            actor_loss    = -self.critic1(states, actor_actions).mean()

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            actor_loss_val        = actor_loss.item()
            self.last_actor_loss  = actor_loss_val

            # Soft-update all target networks (Polyak averaging)
            self._soft_update(self.actor,   self.actor_target)
            self._soft_update(self.critic1, self.critic1_target)
            self._soft_update(self.critic2, self.critic2_target)

        return {
            "critic1_loss": self.last_critic1_loss,
            "critic2_loss": self.last_critic2_loss,
            "actor_loss":   actor_loss_val,
        }

    # ──────────────────────────────────────────────────────────────────────────
    # Utilities
    # ──────────────────────────────────────────────────────────────────────────

    def _soft_update(self, net: nn.Module, target_net: nn.Module) -> None:
        """Polyak averaging: θ_target ← τ·θ + (1-τ)·θ_target"""
        for param, target_param in zip(net.parameters(), target_net.parameters()):
            target_param.data.copy_(
                self.tau * param.data + (1.0 - self.tau) * target_param.data
            )

    def save(self, directory: str = "./saved_models") -> None:
        """Save actor and critic network weights to disk."""
        os.makedirs(directory, exist_ok=True)
        torch.save(self.actor.state_dict(),   os.path.join(directory, "actor.pth"))
        torch.save(self.critic1.state_dict(), os.path.join(directory, "critic1.pth"))
        torch.save(self.critic2.state_dict(), os.path.join(directory, "critic2.pth"))
        print(f"[TD3] Models saved to '{directory}/'")

    def load(self, directory: str = "./saved_models") -> None:
        """Load actor and critic weights from disk."""
        self.actor.load_state_dict(
            torch.load(os.path.join(directory, "actor.pth"),   map_location=self.device)
        )
        self.critic1.load_state_dict(
            torch.load(os.path.join(directory, "critic1.pth"), map_location=self.device)
        )
        self.critic2.load_state_dict(
            torch.load(os.path.join(directory, "critic2.pth"), map_location=self.device)
        )
        # Sync target networks
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic1_target.load_state_dict(self.critic1.state_dict())
        self.critic2_target.load_state_dict(self.critic2.state_dict())
        print(f"[TD3] Models loaded from '{directory}/'")

    @property
    def total_steps(self) -> int:
        return self._train_iterations


In [ ]:
%%writefile ddpg.py
# =============================================================================
# ddpg.py — Deep Deterministic Policy Gradient (DDPG)
#
# Implements the DDPG algorithm from Lillicrap et al., 2015:
#   "Continuous control with deep reinforcement learning"
#
# Key differences from TD3:
#   - Single critic  (no twin critics, no min-trick)
#   - No target policy smoothing noise
#   - Actor updated every step (no policy delay)
#   - Ornstein-Uhlenbeck exploration noise
# =============================================================================

from __future__ import annotations
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import (
    STATE_DIM, ACTION_DIM, HIDDEN_DIM, P_MAX,
    REPLAY_BUFFER_SIZE, MIN_SAMPLES,
    LR_ACTOR, LR_CRITIC,
    GAMMA_DISCOUNT, TAU,
    BATCH_SIZE,
)


# =============================================================================
# Ornstein-Uhlenbeck Noise
# =============================================================================

class OUNoise:
    """
    Ornstein-Uhlenbeck process for temporally correlated exploration noise.
    Commonly used with DDPG for continuous action spaces.

    dx_t = theta*(mu - x_t)*dt + sigma*dW_t
    """

    def __init__(
        self,
        action_dim: int,
        mu:         float = 0.0,
        theta:      float = 0.15,
        sigma:      float = 0.2,
        dt:         float = 1e-2,
    ):
        self.action_dim = action_dim
        self.mu    = mu
        self.theta = theta
        self.sigma = sigma
        self.dt    = dt
        self.reset()

    def reset(self) -> None:
        self.state = np.ones(self.action_dim) * self.mu

    def sample(self) -> np.ndarray:
        dx = (
            self.theta * (self.mu - self.state) * self.dt
            + self.sigma * np.sqrt(self.dt) * np.random.randn(self.action_dim)
        )
        self.state = self.state + dx
        return self.state.copy()

    def set_sigma(self, sigma: float) -> None:
        self.sigma = sigma


# =============================================================================
# Neural Network Modules (identical architecture to TD3)
# =============================================================================

class DDPGActor(nn.Module):
    """
    Policy network π(s) → P_s ∈ [0, P_max].
    Same architecture as TD3 Actor for a fair comparison.
    """

    def __init__(
        self,
        state_dim:  int   = STATE_DIM,
        hidden_dim: int   = HIDDEN_DIM,
        action_dim: int   = ACTION_DIM,
        p_max:      float = P_MAX,
    ):
        super().__init__()
        self.p_max = p_max
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Sigmoid(),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.net(state) * self.p_max


class DDPGCritic(nn.Module):
    """
    Single Q-value network Q(s, a) → ℝ.
    (DDPG uses one critic; TD3 uses two to reduce overestimation bias.)
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        action_dim: int = ACTION_DIM,
        hidden_dim: int = HIDDEN_DIM,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        x = torch.cat([state, action], dim=-1)
        return self.net(x)


# =============================================================================
# DDPG Agent
# =============================================================================

class DDPGAgent:
    """
    Deep Deterministic Policy Gradient agent.

    Maintains:
        actor, actor_target
        critic, critic_target
        OUNoise for exploration
    """

    def __init__(
        self,
        state_dim:    int   = STATE_DIM,
        action_dim:   int   = ACTION_DIM,
        p_max:        float = P_MAX,
        lr_actor:     float = LR_ACTOR,
        lr_critic:    float = LR_CRITIC,
        gamma:        float = GAMMA_DISCOUNT,
        tau:          float = TAU,
        device:       str   = "auto",
    ):
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.p_max  = p_max

        # ── Actor + target ──────────────────────────────────────────────────
        self.actor        = DDPGActor(state_dim, HIDDEN_DIM, action_dim, p_max).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)

        # ── Critic + target ─────────────────────────────────────────────────
        self.critic        = DDPGCritic(state_dim, action_dim, HIDDEN_DIM).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=lr_critic)

        # ── Hyperparameters ──────────────────────────────────────────────────
        self.gamma = gamma
        self.tau   = tau

        # ── Exploration noise ────────────────────────────────────────────────
        self.ou_noise = OUNoise(
            action_dim=action_dim,
            sigma=0.2,
        )

        # Latest losses for logging
        self.last_critic_loss: float       = 0.0
        self.last_actor_loss:  float       = 0.0

        self._train_iterations: int = 0

    # ──────────────────────────────────────────────────────────────────────────
    # Action selection
    # ──────────────────────────────────────────────────────────────────────────

    def select_action(
        self,
        state:             np.ndarray,
        exploration_noise: float = 0.0,
    ) -> float:
        """
        Deterministic action from actor + optional OU exploration noise.

        Args:
            state: (STATE_DIM,) float32 numpy array
            exploration_noise: std scale multiplier for OU noise (0 = greedy)

        Returns:
            Scalar float in [0, P_max].
        """
        with torch.no_grad():
            s      = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            action = self.actor(s).cpu().numpy().flatten()[0]

        if exploration_noise > 0.0:
            # Scale OU noise std proportional to the exploration_noise argument
            self.ou_noise.sigma = exploration_noise
            noise  = self.ou_noise.sample()[0]
            action = action + noise

        return float(np.clip(action, 0.0, self.p_max))

    # ──────────────────────────────────────────────────────────────────────────
    # Training step
    # ──────────────────────────────────────────────────────────────────────────

    def train_step(
        self,
        replay_buffer,
        batch_size: int = BATCH_SIZE,
    ) -> dict | None:
        """
        Perform one DDPG update.

        DDPG update algorithm:
            1. Sample mini-batch from replay buffer
            2. Compute target Q = r + γ(1-done) * Q_tgt(s', μ_tgt(s'))
            3. Update Critic via MSE loss
            4. Update Actor: minimize −Q(s, μ(s))
            5. Soft-update both target networks
        """
        if not replay_buffer.is_ready:
            return None

        self._train_iterations += 1
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        # ── Step 1: Compute target Q-values ─────────────────────────────────
        with torch.no_grad():
            next_actions = self.actor_target(next_states).clamp(0.0, self.p_max)
            q_target     = rewards + self.gamma * (1.0 - dones) * self.critic_target(next_states, next_actions)

        # ── Step 2: Update Critic ────────────────────────────────────────────
        q_current = self.critic(states, actions)
        loss_c    = F.mse_loss(q_current, q_target)
        self.critic_optimizer.zero_grad()
        loss_c.backward()
        self.critic_optimizer.step()
        self.last_critic_loss = loss_c.item()

        # ── Step 3: Update Actor ─────────────────────────────────────────────
        actor_actions = self.actor(states)
        loss_a        = -self.critic(states, actor_actions).mean()
        self.actor_optimizer.zero_grad()
        loss_a.backward()
        self.actor_optimizer.step()
        self.last_actor_loss = loss_a.item()

        # ── Step 4: Soft-update target networks ──────────────────────────────
        self._soft_update(self.actor,  self.actor_target)
        self._soft_update(self.critic, self.critic_target)

        return {
            "critic_loss": self.last_critic_loss,
            "actor_loss":  self.last_actor_loss,
        }

    # ──────────────────────────────────────────────────────────────────────────
    # Utilities
    # ──────────────────────────────────────────────────────────────────────────

    def _soft_update(self, net: nn.Module, target_net: nn.Module) -> None:
        """Polyak averaging: θ_target ← τ·θ + (1-τ)·θ_target"""
        for param, target_param in zip(net.parameters(), target_net.parameters()):
            target_param.data.copy_(
                self.tau * param.data + (1.0 - self.tau) * target_param.data
            )

    def save(self, directory: str = "./saved_models/ddpg") -> None:
        os.makedirs(directory, exist_ok=True)
        torch.save(self.actor.state_dict(),  os.path.join(directory, "ddpg_actor.pth"))
        torch.save(self.critic.state_dict(), os.path.join(directory, "ddpg_critic.pth"))
        print(f"[DDPG] Models saved to '{directory}/'")

    def load(self, directory: str = "./saved_models/ddpg") -> None:
        self.actor.load_state_dict(
            torch.load(os.path.join(directory, "ddpg_actor.pth"),  map_location=self.device)
        )
        self.critic.load_state_dict(
            torch.load(os.path.join(directory, "ddpg_critic.pth"), map_location=self.device)
        )
        self.actor_target  = copy.deepcopy(self.actor)
        self.critic_target = copy.deepcopy(self.critic)
        print(f"[DDPG] Models loaded from '{directory}/'")

    @property
    def total_steps(self) -> int:
        return self._train_iterations


In [ ]:
%%writefile camo_td3.py
# =============================================================================
# camo_td3.py — Constrained Adaptive Multi-Objective TD3 (CAMO-TD3)
#
# Extends TD3 with four novel components:
#   1. Decomposed Multi-Objective Critics — twin critics for each of 3 reward
#      components (throughput, interference penalty, energy cost)
#   2. Adaptive Lagrangian Weights — dual gradient descent to auto-tune the
#      trade-off between objectives during training
#   3. GRU Belief Encoder — recurrent encoder over the last SEQ_LEN
#      observations, producing a compact belief vector concatenated to state
#   4. Directional Exploration Noise — biased noise that nudges actions toward
#      constraint satisfaction (lower power when PU is at risk)
#
# Public API matches TD3Agent / DDPGAgent:
#   select_action(state, exploration_noise) -> float
#   train_step(replay_buffer, batch_size)   -> dict | None
#   save(directory)
#   load(directory)
# =============================================================================

from __future__ import annotations
import os
import math
import numpy as np
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F

from config import (
    STATE_DIM, ACTION_DIM, P_MAX,
    REPLAY_BUFFER_SIZE, MIN_SAMPLES,
    LR_ACTOR, LR_CRITIC,
    GAMMA_DISCOUNT, TAU,
    POLICY_NOISE, NOISE_CLIP, POLICY_DELAY,
    BATCH_SIZE,
    ALPHA, BETA, GAMMA_REWARD, SINR_THRESHOLD,
    # CAMO-specific
    GRU_HIDDEN_SIZE, GRU_NUM_LAYERS, BELIEF_DIM, SEQ_LEN,
    LAMBDA1_INIT, LAMBDA2_INIT, LAMBDA3_INIT, LR_LAMBDA,
    LAMBDA_MIN, LAMBDA_MAX,
    MU_BIAS_INIT, NOISE_DECAY_STEPS, VIOLATION_WINDOW,
    CAMO_HIDDEN_DIM,
)


# =============================================================================
# Sequence Replay Buffer
# =============================================================================

class SequenceReplayBuffer:
    """
    Replay buffer that stores full transitions AND maintains per-episode
    observation histories so we can retrieve the last SEQ_LEN observations
    for the GRU encoder at sample time.

    Storage layout:
        _states[i]      : state at timestep i
        _actions[i]     : action at timestep i
        _rewards_*[i]   : decomposed reward components
        _next_states[i] : next state
        _dones[i]       : done flag
        _obs_histories[i]: (SEQ_LEN, STATE_DIM) tensor of preceding observations
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        action_dim: int = ACTION_DIM,
        seq_len:    int = SEQ_LEN,
        max_size:   int = REPLAY_BUFFER_SIZE,
        device:     str = "cpu",
    ):
        self._max_size = max_size
        self._device   = torch.device(device)
        self._ptr      = 0
        self._size     = 0
        self._seq_len  = seq_len

        # Standard transition storage (on device)
        self._states      = torch.zeros((max_size, state_dim),  dtype=torch.float32, device=self._device)
        self._actions     = torch.zeros((max_size, action_dim), dtype=torch.float32, device=self._device)
        self._next_states = torch.zeros((max_size, state_dim),  dtype=torch.float32, device=self._device)
        self._dones       = torch.zeros((max_size, 1),          dtype=torch.float32, device=self._device)

        # Decomposed reward components
        self._rewards_throughput   = torch.zeros((max_size, 1), dtype=torch.float32, device=self._device)
        self._rewards_interference = torch.zeros((max_size, 1), dtype=torch.float32, device=self._device)
        self._rewards_energy       = torch.zeros((max_size, 1), dtype=torch.float32, device=self._device)

        # Observation histories: (max_size, seq_len, state_dim) — stored on device
        self._obs_histories      = torch.zeros((max_size, seq_len, state_dim), dtype=torch.float32, device=self._device)
        self._next_obs_histories = torch.zeros((max_size, seq_len, state_dim), dtype=torch.float32, device=self._device)

        # Episode-local rolling window (built on CPU, copied per add())
        self._current_history = deque(maxlen=seq_len)

    def reset_episode(self, initial_state: np.ndarray) -> None:
        """Call at the start of each episode to reset the observation history."""
        self._current_history.clear()
        # Pad with copies of the initial state
        for _ in range(self._seq_len):
            self._current_history.append(np.array(initial_state, dtype=np.float32))

    def add(
        self,
        state:      np.ndarray,
        action:     np.ndarray,
        r_throughput:   float,
        r_interference: float,
        r_energy:       float,
        next_state: np.ndarray,
        done:       bool,
    ) -> None:
        p = self._ptr

        # Build observation history tensors
        hist = np.array(list(self._current_history), dtype=np.float32)  # (seq_len, state_dim)
        self._obs_histories[p] = torch.as_tensor(hist, dtype=torch.float32)

        # Advance history with next_state for next_obs_history
        self._current_history.append(np.array(next_state, dtype=np.float32))
        next_hist = np.array(list(self._current_history), dtype=np.float32)
        self._next_obs_histories[p] = torch.as_tensor(next_hist, dtype=torch.float32)

        # Standard fields
        self._states[p]      = torch.as_tensor(state, dtype=torch.float32)
        self._actions[p]     = torch.as_tensor(action, dtype=torch.float32)
        self._next_states[p] = torch.as_tensor(next_state, dtype=torch.float32)
        self._dones[p, 0]    = float(done)

        self._rewards_throughput[p, 0]   = float(r_throughput)
        self._rewards_interference[p, 0] = float(r_interference)
        self._rewards_energy[p, 0]       = float(r_energy)

        self._ptr  = (self._ptr + 1) % self._max_size
        self._size = min(self._size + 1, self._max_size)

    def sample(self, batch_size: int = BATCH_SIZE):
        idx = torch.randint(0, self._size, (batch_size,), device=self._device)
        return (
            self._states[idx],
            self._actions[idx],
            self._rewards_throughput[idx],
            self._rewards_interference[idx],
            self._rewards_energy[idx],
            self._next_states[idx],
            self._dones[idx],
            self._obs_histories[idx],       # (batch, seq_len, state_dim)
            self._next_obs_histories[idx],  # (batch, seq_len, state_dim)
        )

    def __len__(self) -> int:
        return self._size

    @property
    def is_ready(self) -> bool:
        return self._size >= MIN_SAMPLES


# =============================================================================
# GRU Belief Encoder
# =============================================================================

class BeliefEncoder(nn.Module):
    """
    GRU-based encoder that maps a sequence of observations to a compact
    belief vector, capturing temporal channel dynamics.

    Input:  (batch, seq_len, state_dim)
    Output: (batch, belief_dim)
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        hidden_dim: int = GRU_HIDDEN_SIZE,
        num_layers: int = GRU_NUM_LAYERS,
        belief_dim: int = BELIEF_DIM,
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size=state_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.projection = nn.Linear(hidden_dim, belief_dim)

    def forward(self, obs_seq: torch.Tensor) -> torch.Tensor:
        # obs_seq: (batch, seq_len, state_dim)
        _, h_n = self.gru(obs_seq)              # h_n: (num_layers, batch, hidden_dim)
        last_hidden = h_n[-1]                   # (batch, hidden_dim)
        return self.projection(last_hidden)     # (batch, belief_dim)


# =============================================================================
# Actor (augmented state = state + belief)
# =============================================================================

class CAMOActor(nn.Module):
    """
    Policy network: π(s, belief) -> P_s in [0, P_max].
    Takes concatenation of [state, belief_vector].
    """

    def __init__(
        self,
        state_dim:  int   = STATE_DIM,
        belief_dim: int   = BELIEF_DIM,
        hidden_dim: int   = CAMO_HIDDEN_DIM,
        action_dim: int   = ACTION_DIM,
        p_max:      float = P_MAX,
    ):
        super().__init__()
        self.p_max = p_max
        input_dim = state_dim + belief_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Sigmoid(),
        )

    def forward(self, state: torch.Tensor, belief: torch.Tensor) -> torch.Tensor:
        x = torch.cat([state, belief], dim=-1)
        return self.net(x) * self.p_max


# =============================================================================
# Objective-Specific Critic
# =============================================================================

class ObjectiveCritic(nn.Module):
    """
    Q-value network for a single objective: Q_k(s, belief, a) -> R.
    """

    def __init__(
        self,
        state_dim:  int = STATE_DIM,
        belief_dim: int = BELIEF_DIM,
        action_dim: int = ACTION_DIM,
        hidden_dim: int = CAMO_HIDDEN_DIM,
    ):
        super().__init__()
        input_dim = state_dim + belief_dim + action_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        state:  torch.Tensor,
        belief: torch.Tensor,
        action: torch.Tensor,
    ) -> torch.Tensor:
        x = torch.cat([state, belief, action], dim=-1)
        return self.net(x)


# =============================================================================
# CAMO-TD3 Agent
# =============================================================================

class CAMO_TD3Agent:
    """
    Constrained Adaptive Multi-Objective TD3.

    Decomposes the CRN reward into three objectives and learns separate
    twin critics for each. Lagrangian multipliers are adapted online via
    dual gradient descent. A GRU belief encoder captures channel dynamics.
    Directional exploration noise biases actions toward constraint safety.
    """

    def __init__(
        self,
        state_dim:    int   = STATE_DIM,
        action_dim:   int   = ACTION_DIM,
        p_max:        float = P_MAX,
        lr_actor:     float = LR_ACTOR,
        lr_critic:    float = LR_CRITIC,
        gamma:        float = GAMMA_DISCOUNT,
        tau:          float = TAU,
        policy_noise: float = POLICY_NOISE,
        noise_clip:   float = NOISE_CLIP,
        policy_delay: int   = POLICY_DELAY,
        device:       str   = "auto",
    ):
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.p_max  = p_max

        # ── GRU Belief Encoder + target ──────────────────────────────────────
        self.belief_encoder        = BeliefEncoder(state_dim).to(device)
        self.belief_encoder_target = BeliefEncoder(state_dim).to(device)
        self.belief_encoder_target.load_state_dict(self.belief_encoder.state_dict())

        # ── Actor + target ───────────────────────────────────────────────────
        self.actor        = CAMOActor(state_dim, BELIEF_DIM, CAMO_HIDDEN_DIM, action_dim, p_max).to(device)
        self.actor_target = CAMOActor(state_dim, BELIEF_DIM, CAMO_HIDDEN_DIM, action_dim, p_max).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())

        # ── 3 pairs of twin critics (6 total) ───────────────────────────────
        # Objective 0: throughput (R_s)
        # Objective 1: interference penalty (-beta * max(0, thr - SINR_p))
        # Objective 2: energy penalty (-gamma * P_s/P_max)
        self.critics     = nn.ModuleList()
        self.critic_tgts = nn.ModuleList()
        for _ in range(6):
            c = ObjectiveCritic(state_dim, BELIEF_DIM, action_dim, CAMO_HIDDEN_DIM).to(device)
            self.critics.append(c)
            ct = ObjectiveCritic(state_dim, BELIEF_DIM, action_dim, CAMO_HIDDEN_DIM).to(device)
            ct.load_state_dict(c.state_dict())
            self.critic_tgts.append(ct)

        # ── Optimizers ───────────────────────────────────────────────────────
        # Combine belief encoder params with actor for joint optimization
        self.actor_optimizer = torch.optim.Adam(
            list(self.actor.parameters()) + list(self.belief_encoder.parameters()),
            lr=lr_actor,
        )
        self.critic_optimizer = torch.optim.Adam(
            self.critics.parameters(),
            lr=lr_critic,
        )

        # ── Adaptive Lagrangian multipliers (softplus-parameterized) ─────────
        # Initialize so that softplus(log_lambda_k) = LAMBDA_K_INIT exactly.
        # Inverse softplus: x = log(exp(target) - 1)
        def _inv_softplus(t: float) -> float:
            # For small t, exp(t)-1 ≈ t, so inv_softplus(t) ≈ log(t)
            if t > 20.0:
                return t  # softplus(x) ≈ x for large x
            return math.log(math.expm1(t)) if t > 1e-4 else math.log(t)

        self._log_lambda1 = torch.tensor(_inv_softplus(LAMBDA1_INIT),
                                         device=device, requires_grad=True, dtype=torch.float32)
        self._log_lambda2 = torch.tensor(_inv_softplus(LAMBDA2_INIT),
                                         device=device, requires_grad=True, dtype=torch.float32)
        self._log_lambda3 = torch.tensor(_inv_softplus(LAMBDA3_INIT),
                                         device=device, requires_grad=True, dtype=torch.float32)
        self.lambda_optimizer = torch.optim.Adam(
            [self._log_lambda1, self._log_lambda2, self._log_lambda3],
            lr=LR_LAMBDA,
        )

        # ── TD3 hyperparameters ──────────────────────────────────────────────
        self.gamma        = gamma
        self.tau          = tau
        self.policy_noise = policy_noise * p_max
        self.noise_clip   = noise_clip   * p_max
        self.policy_delay = policy_delay

        self._train_iterations = 0

        # ── Directional exploration noise state ──────────────────────────────
        self._mu_bias         = MU_BIAS_INIT
        self._noise_step      = 0
        self._violation_window = deque(maxlen=VIOLATION_WINDOW)

        # ── Inference-time observation history ───────────────────────────────
        self._obs_history = deque(maxlen=SEQ_LEN)

        # Latest losses for logging
        self.last_critic_losses: list[float] = [0.0] * 6
        self.last_actor_loss:   float | None = None

    # ─── Properties ──────────────────────────────────────────────────────────

    @property
    def lambda1(self) -> float:
        return float(F.softplus(self._log_lambda1).item())

    @property
    def lambda2(self) -> float:
        return float(F.softplus(self._log_lambda2).item())

    @property
    def lambda3(self) -> float:
        return float(F.softplus(self._log_lambda3).item())

    @property
    def total_steps(self) -> int:
        return self._train_iterations

    # ─── Episode management ──────────────────────────────────────────────────

    def reset_episode(self, initial_state: np.ndarray) -> None:
        """Reset observation history at the start of each episode."""
        self._obs_history.clear()
        for _ in range(SEQ_LEN):
            self._obs_history.append(np.array(initial_state, dtype=np.float32))

    def record_violation(self, sinr_p: float) -> None:
        """Track whether PU SINR constraint was violated this step."""
        self._violation_window.append(1 if sinr_p < SINR_THRESHOLD else 0)

    # ─── Action selection ────────────────────────────────────────────────────

    def select_action(
        self,
        state:             np.ndarray,
        exploration_noise: float = 0.0,
    ) -> float:
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(self.device)

            # Build observation history tensor
            hist = np.array(list(self._obs_history), dtype=np.float32)
            hist_t = torch.FloatTensor(hist).unsqueeze(0).to(self.device)  # (1, seq_len, state_dim)

            belief = self.belief_encoder(hist_t)  # (1, belief_dim)
            action = self.actor(s, belief).cpu().numpy().flatten()[0]

        # Update history with current observation
        self._obs_history.append(np.array(state, dtype=np.float32))

        if exploration_noise > 0.0:
            # Directional noise: Gaussian with adaptive bias
            self._noise_step += 1
            decay = max(0.0, 1.0 - self._noise_step / NOISE_DECAY_STEPS)

            # Adapt bias: increase negative bias when violations are frequent
            violation_rate = (
                np.mean(list(self._violation_window))
                if len(self._violation_window) > 0
                else 0.0
            )
            adaptive_bias = self._mu_bias * decay * (1.0 + violation_rate)

            noise = np.random.normal(adaptive_bias, exploration_noise)
            action += noise

        return float(np.clip(action, 0.0, self.p_max))

    # ─── Reward decomposition ────────────────────────────────────────────────

    @staticmethod
    def decompose_reward(info: dict) -> tuple[float, float, float]:
        """
        Decompose the scalar CRN reward into its three components using
        the same formula as environment.py.

        Returns (r_throughput, r_interference, r_energy).
        """
        sinr_s = info["sinr_s"]
        sinr_p = info["sinr_p"]
        p_s    = info["p_s"]

        r_throughput   = ALPHA * float(np.log2(1.0 + sinr_s))
        r_interference = -BETA * max(0.0, SINR_THRESHOLD - sinr_p)
        r_energy       = -GAMMA_REWARD * (p_s / P_MAX)

        return r_throughput, r_interference, r_energy

    # ─── Training step ───────────────────────────────────────────────────────

    def train_step(
        self,
        replay_buffer: SequenceReplayBuffer,
        batch_size:    int = BATCH_SIZE,
    ) -> dict | None:
        if not replay_buffer.is_ready:
            return None

        self._train_iterations += 1

        (states, actions,
         r_tput, r_intf, r_energy,
         next_states, dones,
         obs_hist, next_obs_hist) = replay_buffer.sample(batch_size)

        rewards_list = [r_tput, r_intf, r_energy]

        # ── Compute belief vectors ───────────────────────────────────────────
        belief      = self.belief_encoder(obs_hist)
        with torch.no_grad():
            belief_tgt  = self.belief_encoder_target(next_obs_hist)

        # ── Target actions with policy smoothing ─────────────────────────────
        with torch.no_grad():
            noise = torch.randn_like(actions) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            target_actions = (
                self.actor_target(next_states, belief_tgt) + noise
            ).clamp(0.0, self.p_max)

        # ── Update all 6 critics (twin for each of 3 objectives) ────────────
        total_critic_loss = torch.tensor(0.0, device=self.device)
        for obj_idx in range(3):
            c1_idx = obj_idx * 2
            c2_idx = obj_idx * 2 + 1
            reward_k = rewards_list[obj_idx]

            with torch.no_grad():
                q1_tgt = self.critic_tgts[c1_idx](next_states, belief_tgt, target_actions)
                q2_tgt = self.critic_tgts[c2_idx](next_states, belief_tgt, target_actions)
                q_target = reward_k + self.gamma * (1.0 - dones) * torch.min(q1_tgt, q2_tgt)

            q1 = self.critics[c1_idx](states, belief.detach(), actions)
            q2 = self.critics[c2_idx](states, belief.detach(), actions)
            loss_c1 = F.mse_loss(q1, q_target)
            loss_c2 = F.mse_loss(q2, q_target)

            self.last_critic_losses[c1_idx] = loss_c1.item()
            self.last_critic_losses[c2_idx] = loss_c2.item()

            total_critic_loss = total_critic_loss + loss_c1 + loss_c2

        self.critic_optimizer.zero_grad()
        total_critic_loss.backward()
        self.critic_optimizer.step()

        actor_loss_val = None

        # ── Delayed policy + lambda update ───────────────────────────────────
        if self._train_iterations % self.policy_delay == 0:
            # Recompute belief with gradient flow for actor update
            belief_actor = self.belief_encoder(obs_hist)
            actor_actions = self.actor(states, belief_actor)

            # Get Q-values for each objective from critic 1 of each pair
            lam1 = F.softplus(self._log_lambda1)
            lam2 = F.softplus(self._log_lambda2)
            lam3 = F.softplus(self._log_lambda3)

            q_tput   = self.critics[0](states, belief_actor, actor_actions)
            q_intf   = self.critics[2](states, belief_actor, actor_actions)
            q_energy = self.critics[4](states, belief_actor, actor_actions)

            # Scalarized objective: maximize lambda-weighted sum of Q-values
            # Throughput Q is positive (maximize), interference and energy Qs are negative
            actor_loss = -(lam1 * q_tput + lam2 * q_intf + lam3 * q_energy).mean()

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            actor_loss_val       = actor_loss.item()
            self.last_actor_loss = actor_loss_val

            # ── Lagrangian update (dual gradient descent) ────────────────────
            # Goal: increase lambda2 when interference constraint is violated,
            #       decrease when satisfied, to maintain balance
            with torch.no_grad():
                mean_intf_reward = r_intf.mean().item()
                # Constraint: interference penalty should be > some threshold
                # If mean interference reward is very negative, violations are high
                constraint_violation = -mean_intf_reward  # positive = bad

            # Update lambdas: maximize lambda * (constraint_slack)
            # Use negative because optimizer minimizes
            lambda_loss = (
                -self._log_lambda1 * r_tput.mean().detach()
                + self._log_lambda2 * constraint_violation
                - self._log_lambda3 * r_energy.mean().detach()
            )

            self.lambda_optimizer.zero_grad()
            lambda_loss.backward()
            self.lambda_optimizer.step()

            # Clamp lambdas to [LAMBDA_MIN, LAMBDA_MAX] to prevent death spiral
            # We clamp in log-space: log_lambda = log(inv_softplus(clamp(softplus(log_lambda))))
            with torch.no_grad():
                for log_lam in [self._log_lambda1, self._log_lambda2, self._log_lambda3]:
                    lam_val = F.softplus(log_lam)
                    clamped  = lam_val.clamp(LAMBDA_MIN, LAMBDA_MAX)
                    # Invert softplus: log_lam = log(exp(clamped) - 1)
                    new_log = torch.log(torch.expm1(clamped).clamp(min=1e-6))
                    log_lam.copy_(new_log)

            # Soft-update all target networks
            self._soft_update(self.actor, self.actor_target)
            self._soft_update(self.belief_encoder, self.belief_encoder_target)
            for i in range(6):
                self._soft_update(self.critics[i], self.critic_tgts[i])

        return {
            "critic_losses": self.last_critic_losses.copy(),
            "actor_loss":    actor_loss_val,
            "lambda1":       self.lambda1,
            "lambda2":       self.lambda2,
            "lambda3":       self.lambda3,
        }

    # ─── Utilities ───────────────────────────────────────────────────────────

    def _soft_update(self, net: nn.Module, target_net: nn.Module) -> None:
        for param, target_param in zip(net.parameters(), target_net.parameters()):
            target_param.data.copy_(
                self.tau * param.data + (1.0 - self.tau) * target_param.data
            )

    def save(self, directory: str = "./saved_models/camo_td3") -> None:
        os.makedirs(directory, exist_ok=True)
        torch.save(self.actor.state_dict(),
                   os.path.join(directory, "camo_actor.pth"))
        torch.save(self.belief_encoder.state_dict(),
                   os.path.join(directory, "camo_belief_encoder.pth"))
        torch.save(self.critics.state_dict(),
                   os.path.join(directory, "camo_critics.pth"))
        torch.save({
            "log_lambda1": self._log_lambda1,
            "log_lambda2": self._log_lambda2,
            "log_lambda3": self._log_lambda3,
        }, os.path.join(directory, "camo_lambdas.pth"))
        print(f"[CAMO-TD3] Models saved to '{directory}/'")

    def load(self, directory: str = "./saved_models/camo_td3") -> None:
        self.actor.load_state_dict(
            torch.load(os.path.join(directory, "camo_actor.pth"),
                       map_location=self.device)
        )
        self.belief_encoder.load_state_dict(
            torch.load(os.path.join(directory, "camo_belief_encoder.pth"),
                       map_location=self.device)
        )
        self.critics.load_state_dict(
            torch.load(os.path.join(directory, "camo_critics.pth"),
                       map_location=self.device)
        )
        lam = torch.load(os.path.join(directory, "camo_lambdas.pth"),
                         map_location=self.device)
        self._log_lambda1 = lam["log_lambda1"].to(self.device).requires_grad_(True)
        self._log_lambda2 = lam["log_lambda2"].to(self.device).requires_grad_(True)
        self._log_lambda3 = lam["log_lambda3"].to(self.device).requires_grad_(True)

        # Sync targets
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.belief_encoder_target.load_state_dict(self.belief_encoder.state_dict())
        for i in range(6):
            self.critic_tgts[i].load_state_dict(self.critics[i].state_dict())

        print(f"[CAMO-TD3] Models loaded from '{directory}/'")


In [ ]:
%%writefile utils.py
# =============================================================================
# utils.py — Helper utilities: rolling stats, exploration noise, logger
# =============================================================================

import numpy as np
from collections import deque


class RollingStats:
    """Maintains a fixed-size window of scalar values for computing rolling mean."""

    def __init__(self, window: int = 100):
        self._window = window
        self._data: deque[float] = deque(maxlen=window)

    def push(self, value: float) -> None:
        self._data.append(float(value))

    def mean(self) -> float:
        if len(self._data) == 0:
            return 0.0
        return float(np.mean(self._data))

    def values(self) -> list:
        """Returns a list copy of the internal deque."""
        return list(self._data)

    def __len__(self) -> int:
        return len(self._data)


class ExplorationNoise:
    """
    Gaussian exploration noise with linear decay from std_start to std_end
    over decay_steps episodes. After decay_steps the std stays at std_end.
    """

    def __init__(self, std_start: float, std_end: float, decay_steps: int):
        self._std_start  = std_start
        self._std_end    = std_end
        self._decay_steps = max(1, decay_steps)
        self._episode    = 0
        self._current_std = std_start

    def sample(self, shape=(1,)) -> np.ndarray:
        """Draw Gaussian noise with the current std."""
        return np.random.normal(0.0, self._current_std, size=shape)

    def step(self) -> None:
        """Call once at the end of each episode to decay the noise std."""
        self._episode += 1
        frac = min(1.0, self._episode / self._decay_steps)
        self._current_std = self._std_start + frac * (self._std_end - self._std_start)

    @property
    def current_std(self) -> float:
        return self._current_std


class Logger:
    """
    Prints formatted training log rows to console.
    Optionally also writes to a CSV file.
    """

    HEADER = (
        f"{'Episode':>8} | {'Steps':>6} | {'Reward':>9} | {'Avg100':>9} | "
        f"{'SU_Rate':>8} | {'PU_SINR':>8} | {'Power':>7} | {'Buffer':>8}"
    )

    def __init__(self, log_to_file: bool = False, filepath: str = "training_log.csv"):
        self._log_to_file = log_to_file
        self._filepath    = filepath
        self._header_printed = False

        if log_to_file:
            with open(filepath, "w") as f:
                f.write("episode,steps,reward,avg100,su_rate,pu_sinr,power,buffer_size\n")

    def log(
        self,
        episode: int,
        steps: int,
        reward: float,
        avg100: float,
        su_rate: float,
        pu_sinr: float,
        power: float,
        buffer_size: int,
    ) -> None:
        # Print header every 20 episodes for readability
        if not self._header_printed or episode % 20 == 0:
            print("\n" + self.HEADER)
            print("-" * len(self.HEADER))
            self._header_printed = True

        row = (
            f"{episode:>8} | {steps:>6} | {reward:>9.4f} | {avg100:>9.4f} | "
            f"{su_rate:>8.4f} | {pu_sinr:>8.4f} | {power:>7.4f} | {buffer_size:>8}"
        )
        print(row)

        if self._log_to_file:
            with open(self._filepath, "a") as f:
                f.write(
                    f"{episode},{steps},{reward:.6f},{avg100:.6f},"
                    f"{su_rate:.6f},{pu_sinr:.6f},{power:.6f},{buffer_size}\n"
                )


def training_status(episode: int, buffer_size: int, avg100: float) -> str:
    """Returns a human-readable training phase string for the GUI."""
    from config import MIN_SAMPLES, TRAINING_EPISODES
    if buffer_size < MIN_SAMPLES:
        return "Exploring"
    elif episode < TRAINING_EPISODES * 0.2:
        return "Learning"
    elif avg100 > 0.5:
        return "Converging"
    else:
        return "Training"


In [ ]:
%%writefile train_compare_colab.py
# =============================================================================
# train_compare_colab.py — TD3 vs DDPG vs CAMO-TD3 with Checkpoint/Resume
#
# New features over train_compare.py:
#   - save_checkpoint()  : saves agent weights + metrics JSON every N episodes
#   - load_checkpoint()  : finds latest checkpoint and resumes from it
#   - run_algorithm() / run_camo_algorithm() accept start_episode + pre-metrics
# =============================================================================

from __future__ import annotations
import argparse, glob, json, math, os, sys, time
from collections import deque
from dataclasses import dataclass, field
from typing import List

import numpy as np
from scipy.special import erfc

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages

from config import (
    STATE_DIM, ACTION_DIM, P_MAX,
    REPLAY_BUFFER_SIZE, MIN_SAMPLES,
    EXPLORATION_NOISE_STD, EXPLORATION_NOISE_END,
    BATCH_SIZE, SINR_THRESHOLD, NAKAGAMI_M, NAKAGAMI_OMEGA,
    GRAD_UPDATES_PER_STEP,
)
from environment import CRNEnvironment
from td3       import TD3Agent,  ReplayBuffer
from ddpg      import DDPGAgent
from camo_td3  import CAMO_TD3Agent, SequenceReplayBuffer
from utils     import RollingStats, ExplorationNoise

# ── defaults ──────────────────────────────────────────────────────────────────
DEFAULT_EPISODES         = 3000
DEFAULT_STEPS_PER_EP     = 200
DEFAULT_OUTPUT           = "results/crn_comparison_report.pdf"
DEFAULT_CHECKPOINT_EVERY = 300
PRINT_EVERY              = 50
MAX_SCATTER_PTS          = 8000
OUTAGE_WINDOW_SIZE       = 500

TD3_COLOR      = "#1f77b4"
DDPG_COLOR     = "#d62728"
CAMO_TD3_COLOR = "#ff7f0e"
ALPHA_FILL     = 0.15
ALGO_COLORS    = {"TD3": TD3_COLOR, "DDPG": DDPG_COLOR, "CAMO-TD3": CAMO_TD3_COLOR}

# =============================================================================
# Metrics container
# =============================================================================

@dataclass
class RunMetrics:
    name: str
    rewards:        List[float] = field(default_factory=list)
    su_throughputs: List[float] = field(default_factory=list)
    pu_throughputs: List[float] = field(default_factory=list)
    outage_probs:   List[float] = field(default_factory=list)
    avg_bers:       List[float] = field(default_factory=list)
    avg_pu_bers:    List[float] = field(default_factory=list)
    sinr_db_pts:    List[float] = field(default_factory=list)
    ber_pts:        List[float] = field(default_factory=list)
    pu_sinr_db_pts: List[float] = field(default_factory=list)
    pu_ber_pts:     List[float] = field(default_factory=list)
    final_avg_reward:   float = 0.0
    final_avg_su_tput:  float = 0.0
    final_avg_pu_tput:  float = 0.0
    final_outage_prob:  float = 0.0
    final_avg_ber:      float = 0.0
    final_avg_pu_ber:   float = 0.0
    training_time_sec:  float = 0.0


# =============================================================================
# Checkpoint helpers
# =============================================================================

def _metrics_to_dict(m: RunMetrics, episode: int) -> dict:
    return {
        "name": m.name, "episode": episode,
        "rewards":        m.rewards,
        "su_throughputs": m.su_throughputs,
        "pu_throughputs": m.pu_throughputs,
        "outage_probs":   m.outage_probs,
        "avg_bers":       m.avg_bers,
        "avg_pu_bers":    m.avg_pu_bers,
        # cap scatter pts to keep JSON small
        "sinr_db_pts":    m.sinr_db_pts[-2000:],
        "ber_pts":        m.ber_pts[-2000:],
        "pu_sinr_db_pts": m.pu_sinr_db_pts[-2000:],
        "pu_ber_pts":     m.pu_ber_pts[-2000:],
        "training_time_sec": m.training_time_sec,
    }

def _metrics_from_dict(d: dict) -> tuple[RunMetrics, int]:
    m = RunMetrics(name=d["name"])
    m.rewards        = d["rewards"]
    m.su_throughputs = d["su_throughputs"]
    m.pu_throughputs = d["pu_throughputs"]
    m.outage_probs   = d["outage_probs"]
    m.avg_bers       = d["avg_bers"]
    m.avg_pu_bers    = d.get("avg_pu_bers", [])
    m.sinr_db_pts    = d.get("sinr_db_pts", [])
    m.ber_pts        = d.get("ber_pts", [])
    m.pu_sinr_db_pts = d.get("pu_sinr_db_pts", [])
    m.pu_ber_pts     = d.get("pu_ber_pts", [])
    m.training_time_sec = d.get("training_time_sec", 0.0)
    return m, int(d["episode"])


def save_checkpoint(
    algo_name: str,
    episode: int,
    metrics: RunMetrics,
    agent,
    checkpoint_dir: str,
) -> str:
    """Save agent weights + metrics JSON. Returns checkpoint directory path."""
    ep_dir = os.path.join(checkpoint_dir, f"{algo_name.replace('-','_')}_ep{episode:05d}")
    os.makedirs(ep_dir, exist_ok=True)
    agent.save(ep_dir)
    with open(os.path.join(ep_dir, "metrics.json"), "w") as f:
        json.dump(_metrics_to_dict(metrics, episode), f)
    print(f"  ✔ Checkpoint saved → {ep_dir}")
    return ep_dir


def find_latest_checkpoint(algo_name: str, checkpoint_dir: str) -> str | None:
    """Return the path of the latest checkpoint for algo_name, or None."""
    pattern = os.path.join(
        checkpoint_dir,
        f"{algo_name.replace('-','_')}_ep*",
        "metrics.json",
    )
    found = sorted(glob.glob(pattern))
    return os.path.dirname(found[-1]) if found else None


def load_checkpoint(algo_name: str, checkpoint_dir: str):
    """
    Load the latest checkpoint for algo_name.
    Returns (metrics, start_episode, ckpt_path) or (None, 0, None) if none found.
    """
    ckpt_path = find_latest_checkpoint(algo_name, checkpoint_dir)
    if not ckpt_path:
        return None, 0, None
    with open(os.path.join(ckpt_path, "metrics.json")) as f:
        d = json.load(f)
    metrics, start_ep = _metrics_from_dict(d)
    print(f"  ↩ Checkpoint found → {ckpt_path}  (resuming from episode {start_ep})")
    return metrics, start_ep, ckpt_path


# =============================================================================
# Smoothing helpers
# =============================================================================

def smooth(values, window: int = 20) -> np.ndarray:
    if len(values) == 0:
        return np.array([])
    arr = np.array(values, dtype=float)
    if window >= len(arr):
        return arr
    kernel = np.ones(window) / window
    pad = np.pad(arr, (window // 2, window - window // 2 - 1), mode="edge")
    return np.convolve(pad, kernel, mode="valid")


def theoretical_bpsk_ber(snr_db: np.ndarray) -> np.ndarray:
    snr_lin = 10.0 ** (snr_db / 10.0)
    return 0.5 * erfc(np.sqrt(snr_lin))


def nakagami_avg_ber_bpsk(snr_db: np.ndarray, m: float = NAKAGAMI_M) -> np.ndarray:
    snr_lin = 10.0 ** (snr_db / 10.0)
    m_int = int(round(m))
    mu = np.sqrt(snr_lin / (m_int + snr_lin))
    ber = np.zeros_like(snr_lin)
    coeff_base = ((1.0 - mu) / 2.0) ** m_int
    for k in range(m_int):
        binom = math.comb(m_int - 1 + k, k)
        ber += coeff_base * binom * ((1.0 + mu) / 2.0) ** k
    return np.clip(ber, 1e-12, 0.5)


# =============================================================================
# Training loops
# =============================================================================

def run_algorithm(
    algo_name: str,
    agent,
    n_episodes: int,
    steps_per_ep: int,
    verbose: bool = True,
    checkpoint_every: int = 0,
    checkpoint_dir: str = "results",
    resume: bool = True,
) -> RunMetrics:

    # ── Checkpoint resume ─────────────────────────────────────────────────────
    start_ep = 0
    metrics  = RunMetrics(name=algo_name)

    if resume:
        prev_metrics, start_ep, ckpt_path = load_checkpoint(algo_name, checkpoint_dir)
        if prev_metrics is not None:
            metrics = prev_metrics
            agent.load(ckpt_path)
            if start_ep >= n_episodes:
                print(f"  [{algo_name}] Already complete ({start_ep}/{n_episodes} ep). Skipping.")
                _finalize_metrics(metrics, n_episodes)
                return metrics

    env    = CRNEnvironment(steps_per_episode=steps_per_ep)
    buffer = ReplayBuffer(STATE_DIM, ACTION_DIM, REPLAY_BUFFER_SIZE, device=agent.device)
    noise_sched = ExplorationNoise(
        std_start   = EXPLORATION_NOISE_STD * P_MAX,
        std_end     = EXPLORATION_NOISE_END  * P_MAX,
        decay_steps = n_episodes,
    )
    # Fast-forward noise scheduler to current episode
    for _ in range(start_ep):
        noise_sched.step()

    rolling = RollingStats(window=100)
    for r in metrics.rewards:
        rolling.push(r)

    sinr_window    = deque(maxlen=OUTAGE_WINDOW_SIZE)
    scatter_budget = max(1, MAX_SCATTER_PTS // max(n_episodes, 1))
    t_start        = time.perf_counter()

    if verbose:
        remaining = n_episodes - start_ep
        print(f"\n{'='*68}")
        print(f"  [{algo_name}] episodes={n_episodes}  steps/ep={steps_per_ep}  "
              f"start_ep={start_ep}  remaining={remaining}")
        print(f"  Device: {agent.device}  |  Nakagami-m={NAKAGAMI_M}")
        print(f"{'='*68}")

    for ep in range(start_ep, n_episodes):
        state = env.reset()
        ep_reward = 0.0
        ep_sinr_s_list, ep_ber_list = [], []
        ep_rs_list, ep_rp_list      = [], []
        ep_outage_list, ep_pu_ber_list = [], []

        sample_steps = set(np.random.choice(
            steps_per_ep, size=min(scatter_budget, steps_per_ep), replace=False))

        for t in range(steps_per_ep):
            action = agent.select_action(state, noise_sched.current_std)
            result = env.step(action)
            buffer.add(state, np.array([action], dtype=np.float32),
                       result.reward, result.state, result.done)

            sinr_s  = result.info["sinr_s"]; sinr_p = result.info["sinr_p"]
            r_s     = result.info["r_s"]
            r_p     = float(np.log2(1.0 + sinr_p))
            ber     = float(0.5 * erfc(math.sqrt(max(0.0, sinr_s))))
            sinr_db = 10.0 * math.log10(max(1e-9, sinr_s))
            pu_ber      = float(0.5 * erfc(math.sqrt(max(0.0, sinr_p))))
            pu_sinr_db  = 10.0 * math.log10(max(1e-9, sinr_p))

            sinr_window.append(sinr_s)
            ep_sinr_s_list.append(sinr_s); ep_ber_list.append(ber)
            ep_rs_list.append(r_s);        ep_rp_list.append(r_p)
            ep_outage_list.append(1 if sinr_s < SINR_THRESHOLD else 0)
            ep_pu_ber_list.append(pu_ber)

            if t in sample_steps:
                metrics.sinr_db_pts.append(sinr_db); metrics.ber_pts.append(ber)
                metrics.pu_sinr_db_pts.append(pu_sinr_db); metrics.pu_ber_pts.append(pu_ber)

            ep_reward += result.reward
            state      = result.state
            if buffer.is_ready:
                for _ in range(GRAD_UPDATES_PER_STEP):
                    agent.train_step(buffer, BATCH_SIZE)

        noise_sched.step()
        rolling.push(ep_reward)
        metrics.rewards.append(ep_reward)
        metrics.su_throughputs.append(float(np.mean(ep_rs_list)))
        metrics.pu_throughputs.append(float(np.mean(ep_rp_list)))
        metrics.outage_probs.append(float(np.mean(ep_outage_list)))
        metrics.avg_bers.append(float(np.mean(ep_ber_list)))
        metrics.avg_pu_bers.append(float(np.mean(ep_pu_ber_list)))
        metrics.training_time_sec += time.perf_counter() - t_start
        t_start = time.perf_counter()

        if verbose and (ep + 1) % PRINT_EVERY == 0:
            print(f"  ep {ep+1:>5}/{n_episodes}  "
                  f"reward={ep_reward:>8.3f}  avg100={rolling.mean():>8.3f}  "
                  f"outage={metrics.outage_probs[-1]:.3f}  "
                  f"SU_tput={metrics.su_throughputs[-1]:.3f}")

        if checkpoint_every > 0 and (ep + 1) % checkpoint_every == 0:
            save_checkpoint(algo_name, ep + 1, metrics, agent, checkpoint_dir)

    _finalize_metrics(metrics, n_episodes)
    return metrics


def run_camo_algorithm(
    agent: CAMO_TD3Agent,
    n_episodes: int,
    steps_per_ep: int,
    verbose: bool = True,
    checkpoint_every: int = 0,
    checkpoint_dir: str = "results",
    resume: bool = True,
) -> RunMetrics:
    algo_name = "CAMO-TD3"

    start_ep = 0
    metrics  = RunMetrics(name=algo_name)

    if resume:
        prev_metrics, start_ep, ckpt_path = load_checkpoint(algo_name, checkpoint_dir)
        if prev_metrics is not None:
            metrics = prev_metrics
            agent.load(ckpt_path)
            if start_ep >= n_episodes:
                print(f"  [{algo_name}] Already complete ({start_ep}/{n_episodes} ep). Skipping.")
                _finalize_metrics(metrics, n_episodes)
                return metrics

    env    = CRNEnvironment(steps_per_episode=steps_per_ep)
    buffer = SequenceReplayBuffer(STATE_DIM, ACTION_DIM, seq_len=8,
                                  max_size=REPLAY_BUFFER_SIZE, device=agent.device)
    noise_sched = ExplorationNoise(
        std_start   = EXPLORATION_NOISE_STD * P_MAX,
        std_end     = EXPLORATION_NOISE_END  * P_MAX,
        decay_steps = n_episodes,
    )
    for _ in range(start_ep):
        noise_sched.step()

    rolling = RollingStats(window=100)
    for r in metrics.rewards:
        rolling.push(r)

    sinr_window    = deque(maxlen=OUTAGE_WINDOW_SIZE)
    scatter_budget = max(1, MAX_SCATTER_PTS // max(n_episodes, 1))
    t_start        = time.perf_counter()

    if verbose:
        print(f"\n{'='*68}")
        print(f"  [{algo_name}] episodes={n_episodes}  start_ep={start_ep}  "
              f"remaining={n_episodes-start_ep}")
        print(f"  GRU Belief Encoder + 6 Critics + Adaptive Lagrangian")
        print(f"{'='*68}")

    for ep in range(start_ep, n_episodes):
        state = env.reset()
        agent.reset_episode(state)
        buffer.reset_episode(state)

        ep_reward = 0.0
        ep_sinr_s_list, ep_ber_list = [], []
        ep_rs_list, ep_rp_list      = [], []
        ep_outage_list, ep_pu_ber_list = [], []

        sample_steps = set(np.random.choice(
            steps_per_ep, size=min(scatter_budget, steps_per_ep), replace=False))

        for t in range(steps_per_ep):
            action = agent.select_action(state, noise_sched.current_std)
            result = env.step(action)
            r_tput, r_intf, r_energy = CAMO_TD3Agent.decompose_reward(result.info)
            buffer.add(state, np.array([action], dtype=np.float32),
                       r_tput, r_intf, r_energy, result.state, result.done)
            agent.record_violation(result.info["sinr_p"])

            sinr_s  = result.info["sinr_s"]; sinr_p = result.info["sinr_p"]
            r_s     = result.info["r_s"]
            r_p     = float(np.log2(1.0 + sinr_p))
            ber     = float(0.5 * erfc(math.sqrt(max(0.0, sinr_s))))
            sinr_db = 10.0 * math.log10(max(1e-9, sinr_s))
            pu_ber      = float(0.5 * erfc(math.sqrt(max(0.0, sinr_p))))
            pu_sinr_db  = 10.0 * math.log10(max(1e-9, sinr_p))

            sinr_window.append(sinr_s)
            ep_sinr_s_list.append(sinr_s); ep_ber_list.append(ber)
            ep_rs_list.append(r_s);        ep_rp_list.append(r_p)
            ep_outage_list.append(1 if sinr_s < SINR_THRESHOLD else 0)
            ep_pu_ber_list.append(pu_ber)
            if t in sample_steps:
                metrics.sinr_db_pts.append(sinr_db); metrics.ber_pts.append(ber)
                metrics.pu_sinr_db_pts.append(pu_sinr_db); metrics.pu_ber_pts.append(pu_ber)

            ep_reward += result.reward
            state      = result.state
            if buffer.is_ready:
                for _ in range(GRAD_UPDATES_PER_STEP):
                    agent.train_step(buffer, BATCH_SIZE)

        noise_sched.step()
        rolling.push(ep_reward)
        metrics.rewards.append(ep_reward)
        metrics.su_throughputs.append(float(np.mean(ep_rs_list)))
        metrics.pu_throughputs.append(float(np.mean(ep_rp_list)))
        metrics.outage_probs.append(float(np.mean(ep_outage_list)))
        metrics.avg_bers.append(float(np.mean(ep_ber_list)))
        metrics.avg_pu_bers.append(float(np.mean(ep_pu_ber_list)))
        metrics.training_time_sec += time.perf_counter() - t_start
        t_start = time.perf_counter()

        if verbose and (ep + 1) % PRINT_EVERY == 0:
            print(f"  ep {ep+1:>5}/{n_episodes}  "
                  f"reward={ep_reward:>8.3f}  avg100={rolling.mean():>8.3f}  "
                  f"lam=[{agent.lambda1:.2f},{agent.lambda2:.2f},{agent.lambda3:.3f}]")

        if checkpoint_every > 0 and (ep + 1) % checkpoint_every == 0:
            save_checkpoint(algo_name, ep + 1, metrics, agent, checkpoint_dir)

    _finalize_metrics(metrics, n_episodes)
    return metrics


def _finalize_metrics(metrics: RunMetrics, n_episodes: int) -> None:
    tail = min(100, len(metrics.rewards))
    if tail == 0:
        return
    metrics.final_avg_reward  = float(np.mean(metrics.rewards[-tail:]))
    metrics.final_avg_su_tput = float(np.mean(metrics.su_throughputs[-tail:]))
    metrics.final_avg_pu_tput = float(np.mean(metrics.pu_throughputs[-tail:]))
    metrics.final_outage_prob = float(np.mean(metrics.outage_probs[-tail:]))
    metrics.final_avg_ber     = float(np.mean(metrics.avg_bers[-tail:]))
    metrics.final_avg_pu_ber  = float(np.mean(metrics.avg_pu_bers[-tail:]))


# =============================================================================
# PDF report (same as original)
# =============================================================================

def generate_pdf(all_metrics, output_path, n_episodes, steps_per_ep):
    all_metrics = [m for m in all_metrics if m.rewards]
    if not all_metrics:
        print("  No metrics to report — skipping PDF.")
        return
    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)

    plt.rcParams.update({"figure.dpi": 150, "font.size": 11,
                          "axes.titlesize": 13, "axes.labelsize": 12})
    n_ep_actual = max(len(m.rewards) for m in all_metrics)
    episodes = np.arange(1, n_ep_actual + 1)

    def _color(m): return ALGO_COLORS.get(m.name, "#555555")
    def _binned_mean(x, y, lo=-5, hi=25, n_bins=25):
        pts = np.column_stack([x, y])
        bins = np.linspace(lo, hi, n_bins)
        bx, by = [], []
        for i in range(len(bins)-1):
            mask = (pts[:,0]>=bins[i]) & (pts[:,0]<bins[i+1])
            if mask.sum()>0:
                bx.append((bins[i]+bins[i+1])/2); by.append(np.mean(pts[mask,1]))
        return bx, by

    snr_range = np.linspace(-5, 25, 300)
    algo_names = " vs ".join(m.name for m in all_metrics)

    with PdfPages(output_path) as pdf:

        # PAGE 1 — Summary table
        fig, ax = plt.subplots(figsize=(11, 8.5)); ax.axis("off")
        fig.text(0.5, 0.90, f"CRN Power Control: {algo_names}\nNakagami-m Fading Performance Report",
                 ha="center", fontsize=17, fontweight="bold")
        fig.text(0.5, 0.82,
                 f"m={NAKAGAMI_M}  |  Episodes={n_ep_actual}  |  Steps/ep={steps_per_ep}  "
                 f"|  SINR_th={SINR_THRESHOLD}",
                 ha="center", fontsize=10, color="#444")
        col_labels = ["Metric"] + [m.name for m in all_metrics] + ["Winner"]
        rows, fields = [], [
            ("Avg Reward (last 100)",    "final_avg_reward",  "max", ".4f"),
            ("SU Throughput (bits/s/Hz)","final_avg_su_tput", "max", ".4f"),
            ("PU Throughput (bits/s/Hz)","final_avg_pu_tput", "max", ".4f"),
            ("Outage Probability",       "final_outage_prob", "min", ".4f"),
            ("Average BER",              "final_avg_ber",     "min", ".6f"),
            ("Training Time (s)",        "training_time_sec", None,  ".1f"),
        ]
        for label, attr, best_fn, fmt in fields:
            vals = [getattr(m, attr) for m in all_metrics]
            row  = [label] + [f"{v:{fmt}}" for v in vals]
            if best_fn == "max": winner = all_metrics[int(np.argmax(vals))].name
            elif best_fn == "min": winner = all_metrics[int(np.argmin(vals))].name
            else: winner = "-"
            row.append(winner); rows.append(row)
        tbl = ax.table(cellText=rows, colLabels=col_labels, cellLoc="center",
                       loc="center", bbox=[0.02, 0.08, 0.96, 0.62])
        tbl.auto_set_font_size(False); tbl.set_fontsize(10)
        for j in range(len(col_labels)):
            tbl[0,j].set_facecolor("#2c3e50"); tbl[0,j].set_text_props(color="white", fontweight="bold")
        pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # PAGE 2 — SU SINR vs BER
        fig, ax = plt.subplots(figsize=(10,6))
        ax.semilogy(snr_range, theoretical_bpsk_ber(snr_range), "k--", lw=1.4, label="BPSK AWGN", alpha=0.7)
        ax.semilogy(snr_range, nakagami_avg_ber_bpsk(snr_range, NAKAGAMI_M), "k-.",
                    lw=1.4, label=f"Nakagami-m={int(NAKAGAMI_M)}", alpha=0.7)
        for m in all_metrics:
            c = _color(m)
            if m.sinr_db_pts:
                ax.scatter(m.sinr_db_pts, m.ber_pts, color=c, alpha=0.3, s=10, label=f"{m.name} (sim)")
                bx, by = _binned_mean(m.sinr_db_pts, m.ber_pts)
                if bx: ax.semilogy(bx, by, color=c, lw=2.2, marker="o", ms=4, label=f"{m.name} mean")
        ax.set(xlabel="SINR (dB)", ylabel="BER", xlim=(-5,25), ylim=(1e-6,0.6),
               title="SU SINR vs BER — Nakagami-m Fading")
        ax.legend(loc="lower left"); ax.grid(True, which="both", alpha=0.3)
        fig.tight_layout(); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # PAGE 3 — PU SINR vs BER
        fig, ax = plt.subplots(figsize=(10,6))
        ax.semilogy(snr_range, nakagami_avg_ber_bpsk(snr_range, NAKAGAMI_M), "k-.",
                    lw=1.4, label=f"Nakagami-m={int(NAKAGAMI_M)}", alpha=0.7)
        for m in all_metrics:
            c = _color(m)
            if m.pu_sinr_db_pts:
                ax.scatter(m.pu_sinr_db_pts, m.pu_ber_pts, color=c, alpha=0.25, s=10, label=f"{m.name} PU")
                bx, by = _binned_mean(m.pu_sinr_db_pts, m.pu_ber_pts)
                if bx: ax.semilogy(bx, by, color=c, lw=2.2, marker="s", ms=4)
        ax.axvline(x=10*np.log10(SINR_THRESHOLD), color="green", ls="--", lw=1.4, label="PU threshold")
        ax.set(xlabel="PU SINR (dB)", ylabel="BER", xlim=(-5,25), ylim=(1e-6,0.6),
               title="PU SINR vs BER — Effect of SU Interference")
        ax.legend(loc="lower left"); ax.grid(True, which="both", alpha=0.3)
        fig.tight_layout(); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # PAGES 4-6 — SU Tput, PU Tput, Outage
        for data_key, ylabel, title, hline in [
            ("su_throughputs", "Throughput (bits/s/Hz)", "SU Throughput vs Episodes", None),
            ("pu_throughputs", "Throughput (bits/s/Hz)", "PU Throughput vs Episodes", None),
            ("outage_probs",   "Outage Probability",      "Outage Probability vs Episodes", 0.05),
        ]:
            fig, ax = plt.subplots(figsize=(10, 5.5))
            for m in all_metrics:
                c = _color(m); raw = np.array(getattr(m, data_key))
                ep_x = np.arange(1, len(raw)+1)
                sm = smooth(raw, window=20)
                ax.plot(ep_x, raw, color=c, alpha=0.2, lw=0.7)
                ax.plot(ep_x, sm,  color=c, lw=2.0, label=m.name)
                ax.fill_between(ep_x, np.maximum(0, sm-raw.std()), sm+raw.std(), color=c, alpha=ALPHA_FILL)
            if hline: ax.axhline(hline, color="gray", ls="--", lw=1.2, label="5% target"); ax.set_ylim(0,1)
            ax.set(xlabel="Episode", ylabel=ylabel, title=title)
            ax.legend(); ax.grid(True, alpha=0.3)
            fig.tight_layout(); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # PAGE 7 — Individual reward curves
        n_algos = len(all_metrics)
        cols = min(n_algos, 3); rows_g = math.ceil(n_algos/cols)
        fig, axes = plt.subplots(rows_g, cols, figsize=(5*cols, 5*rows_g), squeeze=False)
        for idx, m in enumerate(all_metrics):
            ax = axes[idx//cols][idx%cols]; c = _color(m)
            raw = np.array(m.rewards); ep_x = np.arange(1, len(raw)+1)
            ax.plot(ep_x, raw, color=c, alpha=0.2, lw=0.7)
            ax.plot(ep_x, smooth(raw,20), color=c, lw=2.2, label="Smoothed")
            ax.set(xlabel="Episode", ylabel="Episode Reward", title=f"{m.name} Reward Curve")
            ax.legend(); ax.grid(True, alpha=0.3)
        for idx in range(n_algos, rows_g*cols):
            axes[idx//cols][idx%cols].set_visible(False)
        fig.suptitle("Episode Reward Curves", fontsize=14); fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        # PAGE 8 — Overlay reward comparison
        fig, ax = plt.subplots(figsize=(10, 5.5))
        for m in all_metrics:
            c = _color(m); raw = np.array(m.rewards); ep_x = np.arange(1, len(raw)+1)
            ax.plot(ep_x, raw, color=c, alpha=0.18, lw=0.7)
            ax.plot(ep_x, smooth(raw,20), color=c, lw=2.2, label=m.name)
        ax.set(xlabel="Episode", ylabel="Episode Reward", title=f"{algo_names} — Reward Comparison")
        ax.legend(); ax.grid(True, alpha=0.3)
        fig.tight_layout(); pdf.savefig(fig, bbox_inches="tight"); plt.close(fig)

        d = pdf.infodict()
        d["Title"] = f"CRN {algo_names} Report"

    print(f"\n  PDF report saved: {os.path.abspath(output_path)}")


# =============================================================================
# Entry point
# =============================================================================

VALID_AGENTS = {"td3", "ddpg", "camo-td3"}
DISPLAY      = {"td3": "TD3", "ddpg": "DDPG", "camo-td3": "CAMO-TD3"}

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--episodes",          type=int,  default=DEFAULT_EPISODES)
    p.add_argument("--steps-per-ep",      type=int,  default=DEFAULT_STEPS_PER_EP)
    p.add_argument("--output",            type=str,  default=DEFAULT_OUTPUT)
    p.add_argument("--seed",              type=int,  default=42)
    p.add_argument("--agents",            type=str,  default="td3,ddpg,camo-td3")
    p.add_argument("--checkpoint-every",  type=int,  default=DEFAULT_CHECKPOINT_EVERY)
    p.add_argument("--checkpoint-dir",    type=str,  default="results/checkpoints")
    p.add_argument("--no-resume",         action="store_true",
                   help="Ignore existing checkpoints and train from scratch")
    return p.parse_args()


def main():
    args = parse_args()
    agent_list = [a.strip().lower() for a in args.agents.split(",")]
    resume = not args.no_resume

    np.random.seed(args.seed)
    import torch; torch.manual_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    os.makedirs(os.path.dirname(args.output) if os.path.dirname(args.output) else ".", exist_ok=True)

    print(f"\n{'='*68}")
    print(f"  CRN Comparison: {' vs '.join(DISPLAY[a] for a in agent_list)}")
    print(f"  Episodes={args.episodes} | Steps/ep={args.steps_per_ep} | "
          f"Checkpoint every {args.checkpoint_every} ep")
    print(f"  Resume={resume} | Checkpoint dir: {args.checkpoint_dir}")
    print(f"{'='*68}")

    all_metrics = []
    for i, key in enumerate(agent_list):
        np.random.seed(args.seed + i)
        import torch; torch.manual_seed(args.seed + i)
        name = DISPLAY[key]
        if key == "td3":
            agent = TD3Agent()
            m = run_algorithm(name, agent, args.episodes, args.steps_per_ep,
                              checkpoint_every=args.checkpoint_every,
                              checkpoint_dir=args.checkpoint_dir,
                              resume=resume)
        elif key == "ddpg":
            agent = DDPGAgent()
            m = run_algorithm(name, agent, args.episodes, args.steps_per_ep,
                              checkpoint_every=args.checkpoint_every,
                              checkpoint_dir=args.checkpoint_dir,
                              resume=resume)
        elif key == "camo-td3":
            agent = CAMO_TD3Agent()
            m = run_camo_algorithm(agent, args.episodes, args.steps_per_ep,
                                   checkpoint_every=args.checkpoint_every,
                                   checkpoint_dir=args.checkpoint_dir,
                                   resume=resume)
        all_metrics.append(m)

    print("\n  Generating PDF report...")
    generate_pdf(all_metrics, args.output, args.episodes, args.steps_per_ep)
    print("\n  All done!")

if __name__ == "__main__":
    main()


## Step 4 — Training Configuration

Edit the variables below before running. `RESUME = True` (default) will automatically pick up from the last checkpoint.

In [ ]:
# ── Training Configuration ───────────────────────────────────────────
EPISODES         = 3000    # Total episodes per algorithm
STEPS_PER_EP     = 200     # Steps per episode
CHECKPOINT_EVERY = 300     # Save checkpoint every N episodes
AGENTS           = 'td3,ddpg,camo-td3'   # Comma-separated: td3 / ddpg / camo-td3
SEED             = 42
RESUME           = True    # Set False to train from scratch (ignores checkpoints)

import os
# Paths (checkpoints go to Drive, report too)
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
REPORT_PATH    = os.path.join(DRIVE_DIR, 'crn_comparison_report.pdf')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Episodes     : {EPISODES}')
print(f'Checkpoint   : every {CHECKPOINT_EVERY} episodes → {CHECKPOINT_DIR}')
print(f'Resume       : {RESUME}')
print(f'Report path  : {REPORT_PATH}')

# Show existing checkpoints if any
existing = sorted(os.listdir(CHECKPOINT_DIR)) if os.path.isdir(CHECKPOINT_DIR) else []
if existing:
    print(f'\nExisting checkpoints ({len(existing)}):')
    for c in existing: print(f'  {c}')
else:
    print('\nNo checkpoints found — starting fresh.')


## Step 5 — Run Training

Runs TD3 → DDPG → CAMO-TD3 sequentially. If `RESUME=True` and checkpoints exist, each algorithm picks up exactly where it left off.

**Estimated time on T4 GPU:** ~3–4 hours for all three algorithms at 3000 episodes.

In [ ]:
import subprocess, sys

resume_flag = [] if RESUME else ['--no-resume']

cmd = [
    sys.executable, 'train_compare_colab.py',
    '--episodes',         str(EPISODES),
    '--steps-per-ep',     str(STEPS_PER_EP),
    '--agents',           AGENTS,
    '--output',           REPORT_PATH,
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--checkpoint-dir',   CHECKPOINT_DIR,
    '--seed',             str(SEED),
] + resume_flag

print('Command:', ' '.join(cmd))
print('Starting training...\n')
result = subprocess.run(cmd)
print('\nReturn code:', result.returncode)


## Step 6 — View Results & Download Report

In [ ]:
import os
from IPython.display import FileLink, display

if os.path.exists(REPORT_PATH):
    size_mb = os.path.getsize(REPORT_PATH) / 1e6
    print(f'✔ Report ready: {REPORT_PATH}  ({size_mb:.2f} MB)')
    # Copy to /content for easy download
    local = '/content/crn_comparison_report.pdf'
    import shutil; shutil.copy(REPORT_PATH, local)
    display(FileLink(local, result_html_prefix='⬇ Click to download: '))
else:
    print('Report not found — check the training output above for errors.')

# List all checkpoints
print('\nCheckpoints saved:')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    p = os.path.join(CHECKPOINT_DIR, f)
    if os.path.isdir(p):
        files = os.listdir(p)
        print(f'  {f}/  ({len(files)} files)')


## Step 7 — Quick Inline Reward Plot (optional)

In [ ]:
import json, glob, numpy as np, matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
colors = {'TD3': '#1f77b4', 'DDPG': '#d62728', 'CAMO_TD3': '#ff7f0e'}

for algo_color_key, color in colors.items():
    # Find latest metrics.json for this algo
    pattern = os.path.join(CHECKPOINT_DIR, f'{algo_color_key}_ep*', 'metrics.json')
    found = sorted(glob.glob(pattern))
    if not found: continue
    with open(found[-1]) as f:
        d = json.load(f)
    rewards = np.array(d['rewards'])
    ep = np.arange(1, len(rewards)+1)
    sm = np.convolve(rewards, np.ones(20)/20, mode='same')
    ax.plot(ep, rewards, color=color, alpha=0.2, lw=0.6)
    ax.plot(ep, sm, color=color, lw=2.2, label=d['name'])

ax.set(xlabel='Episode', ylabel='Episode Reward',
       title='Reward Curves — CRN Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
